# 02. Признаки и базовая модель (этап 2)
1. ≥ 10 новых признаков (15) с экономическим обоснованием.
2. Классическая скоринговая карта: WoE-биннинг → логистическая регрессия → баллы.
3. Метрики базовой модели на validation — планка для бустинга.

In [ ]:
import os, sys, warnings
warnings.filterwarnings("ignore")
root = os.getcwd()                              # ищем корень проекта (где лежит config.yaml) — работает и в Colab
while not os.path.exists(os.path.join(root, "config.yaml")) and root != os.path.dirname(root):
    root = os.path.dirname(root)
sys.path.insert(0, root); os.chdir(root)
import pandas as pd, numpy as np
pd.set_option("display.max_columns", 30, "display.width", 160, "display.float_format", "{:.4f}".format)
from src import pipeline as P
c = P.context()

## 1. Новые признаки и их экономическое обоснование

In [ ]:
from src.features import feature_rationales, NEW_FEATURES
rat = feature_rationales()
Xtr, ytr = c["Xtr"], c["ytr"]
from src import metrics as M
tab = pd.DataFrame({"feature": NEW_FEATURES,
                    "gini_train": [abs(M.gini(ytr, Xtr[f])) for f in NEW_FEATURES],
                    "rationale": [rat[f] for f in NEW_FEATURES]})
print(len(NEW_FEATURES), "новых признаков")
pd.set_option("display.max_colwidth", 400)
tab

## 2. Базовая модель: скоринговая карта
* Квантильный биннинг (до 10 корзин, слияние соседних до ≥5% наблюдений и ≥5 банкротов в корзине), WoE со сглаживанием.
* Отбор: IV ≥ 0,05, жадное удаление коррелированных WoE (|ρ| ≥ 0,7), не более 15 признаков; признаки с «неправильным» знаком коэффициента удаляются.
* Логистическая регрессия (L2); регуляризация `C` выбрана по TimeSeriesSplit на train (не KFold).
* Баллы: base score 600 при odds 50:1, PDO = 20.

In [ ]:
r2 = P.ensure("step2")
print("C =", r2["best_C"], "| признаков в карте:", r2["n_features"], "| знаки коэффициентов корректны:", r2["sign_ok"])
pd.DataFrame(r2["grid"])

In [ ]:
iv = pd.read_csv("artifacts/iv_table.csv"); iv.head(20)

In [ ]:
pts = pd.read_csv("artifacts/scorecard_points.csv")
pts[["feature", "bin", "lo", "hi", "n", "bads", "bad_rate", "woe", "points"]].head(30)

## 3. Метрики на validation — планка для бустинга

In [ ]:
pd.DataFrame({"train": r2["train"], "validation": r2["valid"], "validation (isotonic)": r2["valid_calibrated"]}).T

In [ ]:
print("95% ДИ Gini на validation:", [round(x, 3) for x in r2["valid_gini_ci"]], "(в validation мало банкротов → интервал широкий)")
print("CV-Gini (TimeSeriesSplit на train): %.3f ± %.3f" % (r2["cv_gini"], r2["cv_gini_std"]))

**Вывод.** Скоринговая карта (15 признаков, C = 0,01) даёт Gini 0,969 на validation и 0,795 по TimeSeriesSplit на train. Среди отобранных признаков самые сильные по IV — retained_self_financing, Retained Earnings to Total Assets и interest_drag; все коэффициенты имеют ожидаемый знак. Оценка на validation шумная (13 банкротов, ДИ 0,946–0,987), поэтому ориентир — результат по TimeSeriesSplit. Эти значения — планка для бустинга.